### Modul 3 - Tugas Praktikum Demo

#### Import Library

In [54]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

In [55]:
def configure_pandas_display():
    pd.set_option('display.max_colwidth', 50)
    pd.set_option('display.width', 1000)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.float_format', '{:.4f}'.format)
    pd.set_option('display.colheader_justify', 'center')
    pd.set_option('display.precision', 4)
    pd.set_option('display.max_rows', 100)

configure_pandas_display()

#### Load Dataset

In [ ]:
df = pd.read_csv('accident_dataset.csv')
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nSample data:")
print(df.head())

Dataset shape: (12316, 32)

Data types:
Time                           object
Day_of_week                    object
Age_band_of_driver             object
Sex_of_driver                  object
Educational_level              object
Vehicle_driver_relation        object
Driving_experience             object
Type_of_vehicle                object
Owner_of_vehicle               object
Service_year_of_vehicle        object
Defect_of_vehicle              object
Area_accident_occured          object
Lanes_or_Medians               object
Road_allignment                object
Types_of_Junction              object
Road_surface_type              object
Road_surface_conditions        object
Light_conditions               object
Weather_conditions             object
Type_of_collision              object
Number_of_vehicles_involved     int64
Number_of_casualties            int64
Vehicle_movement               object
Casualty_class                 object
Sex_of_casualty                object
Age_band_o

#### Step 1 : Handling Categorical Values

In [57]:
print("\n--- Step 1: Handling Categorical Values ---")


--- Step 1: Handling Categorical Values ---


#### Identify categorical columns


In [58]:
categorical_cols = df.select_dtypes(include=['object']).columns
print("Categorical columns:", list(categorical_cols))

Categorical columns: ['Time', 'Day_of_week', 'Age_band_of_driver', 'Sex_of_driver', 'Educational_level', 'Vehicle_driver_relation', 'Driving_experience', 'Type_of_vehicle', 'Owner_of_vehicle', 'Service_year_of_vehicle', 'Defect_of_vehicle', 'Area_accident_occured', 'Lanes_or_Medians', 'Road_allignment', 'Types_of_Junction', 'Road_surface_type', 'Road_surface_conditions', 'Light_conditions', 'Weather_conditions', 'Type_of_collision', 'Vehicle_movement', 'Casualty_class', 'Sex_of_casualty', 'Age_band_of_casualty', 'Casualty_severity', 'Work_of_casuality', 'Fitness_of_casuality', 'Pedestrian_movement', 'Cause_of_accident', 'Accident_severity']


#### Method 1: One-Hot Encoding for 'Day_of_week'

In [59]:
print("\nApplying One-Hot Encoding to 'Day_of_week'")
day_dummies = pd.get_dummies(df['Day_of_week'], prefix='day')
df = pd.concat([df, day_dummies], axis=1)
print(day_dummies.head)


Applying One-Hot Encoding to 'Day_of_week'
<bound method NDFrame.head of        day_Friday  day_Monday  day_Saturday  day_Sunday  day_Thursday  day_Tuesday  day_Wednesday
0         False        True        False        False        False        False         False    
1         False        True        False        False        False        False         False    
2         False        True        False        False        False        False         False    
3         False       False        False         True        False        False         False    
4         False       False        False         True        False        False         False    
...           ...         ...           ...         ...           ...          ...            ...
12311     False       False        False        False        False        False          True    
12312     False       False        False         True        False        False         False    
12313     False       False        False    

#### Method 2: Label Encoding for 'Sex_of_driver'

In [60]:
print("Applying Label Encoding to 'Sex_of_driver'")
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Sex_of_driver_encoded'] = le.fit_transform(df['Sex_of_driver'])
print("Encoding map:", dict(zip(le.classes_, le.transform(le.classes_))))
print(df[['Sex_of_driver', 'Sex_of_driver_encoded']].tail())

Applying Label Encoding to 'Sex_of_driver'
Encoding map: {'Female': np.int64(0), 'Male': np.int64(1), 'Unknown': np.int64(2)}
      Sex_of_driver  Sex_of_driver_encoded
12311       Male               1          
12312       Male               1          
12313       Male               1          
12314     Female               0          
12315       Male               1          


#### Method 3: Ordinal Encoding for 'Educational_level'

In [61]:
print("Applying Ordinal Encoding to 'Educational_level'")
education_order = {
    'Below school': 0,
    'Elementary school': 1,
    'Junior high school': 2,
    'High school': 3,
    'Above high school': 4,
    'Unknown': -1
}
df['Education_level_encoded'] = df['Educational_level'].map(education_order)
print(df[['Educational_level', 'Education_level_encoded']].head())


Applying Ordinal Encoding to 'Educational_level'
   Educational_level   Education_level_encoded
0   Above high school          4.0000         
1  Junior high school          2.0000         
2  Junior high school          2.0000         
3  Junior high school          2.0000         
4  Junior high school          2.0000         


#### Applying Binning to 'Time' Column

In [62]:
if 'Time' in df.columns:
    print("\nApplying Binning to 'Time'")
    bins = [0, 6, 12, 18, 24]
    labels = ['Pagi', 'Siang', 'Sore', 'Malam']

    df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.hour
    
    df['Time_Category'] = pd.cut(df['Hour'], bins=bins, labels=labels, include_lowest=True)
    print(df[['Time', 'Time_Category']].head(25))
    
    df = df.drop('Hour', axis=1)
else:
    print("\n'Time' column not found in the dataset. Skipping binning step.")


Applying Binning to 'Time'
      Time   Time_Category
0   17:02:00       Sore   
1   17:02:00       Sore   
2   17:02:00       Sore   
3    1:06:00       Pagi   
4    1:06:00       Pagi   
5   14:15:00       Sore   
6   17:30:00       Sore   
7   17:20:00       Sore   
8   17:20:00       Sore   
9   17:20:00       Sore   
10  14:40:00       Sore   
11  14:40:00       Sore   
12  17:45:00       Sore   
13  17:45:00       Sore   
14  17:45:00       Sore   
15  22:45:00      Malam   
16  22:45:00      Malam   
17  22:45:00      Malam   
18  22:45:00      Malam   
19   8:20:00      Siang   
20   8:20:00      Siang   
21  15:10:00       Sore   
22  12:11:00      Siang   
23  12:11:00      Siang   
24  18:36:00       Sore   


### Step 2: Data Normalization


In [63]:
print("\n--- Step 2: Data Normalization ---")


--- Step 2: Data Normalization ---


#### Min-Max Scaling on Number of Vehicles

In [64]:
print("Applying Min-Max Scaling to 'Number_of_vehicles_involved'")
scaler_minmax = MinMaxScaler()
df['Vehicles_minmax_scaled'] = scaler_minmax.fit_transform(df[['Number_of_vehicles_involved']])
print(df[['Number_of_vehicles_involved', 'Vehicles_minmax_scaled']].head())

Applying Min-Max Scaling to 'Number_of_vehicles_involved'
   Number_of_vehicles_involved  Vehicles_minmax_scaled
0               2                       0.1667        
1               2                       0.1667        
2               2                       0.1667        
3               2                       0.1667        
4               2                       0.1667        


#### Z-Score Scaling on Number of Casualties

In [65]:
print("Applying Z-Score Scaling to 'Number_of_casualties'")
scaler_z = StandardScaler()
df['Casualties_z_scaled'] = scaler_z.fit_transform(df[['Number_of_casualties']])
print(df[['Number_of_casualties', 'Casualties_z_scaled']].head())

Applying Z-Score Scaling to 'Number_of_casualties'
   Number_of_casualties  Casualties_z_scaled
0            2                 0.4486       
1            2                 0.4486       
2            2                 0.4486       
3            2                 0.4486       
4            2                 0.4486       


#### Decimal Scaling for Longitude and Latitude

In [ ]:
if 'Casualty_severity' in df.columns:
    print("Applying Decimal Scaling to 'Casualty_severity'")
    
    def decimal_scaling(series):
        numeric_series = pd.to_numeric(series, errors='coerce')
        max_abs_val = numeric_series.abs().max()
        if pd.isna(max_abs_val) or max_abs_val == 0:
            return numeric_series
        
        digits = int(np.ceil(np.log10(max_abs_val)))
        
        return numeric_series / (10 ** digits)
    
    df['Casualty_severity_decimal_scaled'] = decimal_scaling(df['Casualty_severity'])
    
    print(df)
else:
    print("\n'Casualty_severity' column not found in the dataset. Skipping decimal scaling step.")

Applying Decimal Scaling to 'Casualty_severity'
         Time   Day_of_week Age_band_of_driver Sex_of_driver  Educational_level  Vehicle_driver_relation Driving_experience   Type_of_vehicle    Owner_of_vehicle Service_year_of_vehicle Defect_of_vehicle Area_accident_occured                  Lanes_or_Medians                                Road_allignment                 Types_of_Junction Road_surface_type Road_surface_conditions    Light_conditions    Weather_conditions            Type_of_collision              Number_of_vehicles_involved  Number_of_casualties Vehicle_movement  Casualty_class  Sex_of_casualty Age_band_of_casualty Casualty_severity Work_of_casuality Fitness_of_casuality                Pedestrian_movement                           Cause_of_accident           Accident_severity  day_Friday  day_Monday  day_Saturday  day_Sunday  day_Thursday  day_Tuesday  day_Wednesday  Sex_of_driver_encoded  Education_level_encoded Time_Category  Vehicles_minmax_scaled  Casualties_z_scaled  

### Step 3: Dimensionality Reduction

In [67]:
print("\n--- Step 3: Dimensionality Reduction ---")


--- Step 3: Dimensionality Reduction ---


#### Prepare numerical features for correlation analysis

In [68]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
numerical_df = df[numerical_cols]
print(numerical_df)

       Number_of_vehicles_involved  Number_of_casualties  Sex_of_driver_encoded  Education_level_encoded  Vehicles_minmax_scaled  Casualties_z_scaled  Casualty_severity_decimal_scaled
0                   2                         2                     1                    4.0000                   0.1667                0.4486                         NaN             
1                   2                         2                     1                    2.0000                   0.1667                0.4486                         NaN             
2                   2                         2                     1                    2.0000                   0.1667                0.4486                      0.3000             
3                   2                         2                     1                    2.0000                   0.1667                0.4486                      0.3000             
4                   2                         2                     1           

#### Feature Selection - correlation with Number of Casualties

In [69]:
if 'Number_of_casualties' in numerical_df.columns:
    print("Feature Selection based on correlation with 'Number_of_casualties'")
    correlation = numerical_df.corr()['Number_of_casualties'].sort_values(ascending=False)
    print(correlation)
    
    top_features = correlation[1:6].index.tolist()
    print("Top 5 correlated features:", top_features)
    
    selected_features_df = df[top_features]

Feature Selection based on correlation with 'Number_of_casualties'
Number_of_casualties                1.0000
Casualties_z_scaled                 1.0000
Vehicles_minmax_scaled              0.2134
Number_of_vehicles_involved         0.2134
Sex_of_driver_encoded               0.0485
Casualty_severity_decimal_scaled    0.0077
Education_level_encoded            -0.0166
Name: Number_of_casualties, dtype: float64
Top 5 correlated features: ['Casualties_z_scaled', 'Vehicles_minmax_scaled', 'Number_of_vehicles_involved', 'Sex_of_driver_encoded', 'Casualty_severity_decimal_scaled']


#### Feature Extraction using PCA

In [70]:
print("\nFeature Extraction using PCA")
pca_columns = [col for col in numerical_cols if not col.endswith(('_scaled', '_encoded'))]
if len(pca_columns) >= 5:
    pca_data = df[pca_columns].fillna(0) 
    
    pca = PCA(n_components=5)
    pca_result = pca.fit_transform(pca_data)
    
    pca_df = pd.DataFrame(
        pca_result, 
        columns=[f'PCA_{i+1}' for i in range(5)],
        index=df.index
    )
    
    df = pd.concat([df, pca_df], axis=1)
    
    print("Explained Variance Ratio:", pca.explained_variance_ratio_)
    print("Cumulative Explained Variance:", sum(pca.explained_variance_ratio_))
else:
    print(f"Not enough numerical features for PCA. Found only {len(pca_columns)} features.")



Feature Extraction using PCA
Not enough numerical features for PCA. Found only 2 features.


### Step 4: Data Splitting

In [71]:
print("\n--- Step 4: Data Splitting ---")


--- Step 4: Data Splitting ---


#### Assuming 'Accident_severity' is the target variable

In [72]:
X = df.drop(['Accident_severity'], axis=1)
y = df['Accident_severity']

#### First split: 70% train, 30% temp

In [73]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

#### Second split: 15% validation, 15% test (from the 30% temp)

In [74]:
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [75]:
print(f"Training set size: {X_train.shape[0]} ({X_train.shape[0]/len(df):.1%})")
print(f"Validation set size: {X_val.shape[0]} ({X_val.shape[0]/len(df):.1%})")
print(f"Test set size: {X_test.shape[0]} ({X_test.shape[0]/len(df):.1%})")

# Check the distribution of the target variable in each set
print("\nTarget Distribution:")
print("Original:", y.value_counts(normalize=True))
print("\nTraining:", y_train.value_counts(normalize=True))
print("\nValidation:", y_val.value_counts(normalize=True))
print("\nTest:", y_test.value_counts(normalize=True))

Training set size: 8621 (70.0%)
Validation set size: 1847 (15.0%)
Test set size: 1848 (15.0%)

Target Distribution:
Original: Accident_severity
Slight Injury    0.8456
Serious Injury   0.1415
Fatal injury     0.0128
Name: proportion, dtype: float64

Training: Accident_severity
Slight Injury    0.8456
Serious Injury   0.1415
Fatal injury     0.0129
Name: proportion, dtype: float64

Validation: Accident_severity
Slight Injury    0.8457
Serious Injury   0.1413
Fatal injury     0.0130
Name: proportion, dtype: float64

Test: Accident_severity
Slight Injury    0.8458
Serious Injury   0.1418
Fatal injury     0.0124
Name: proportion, dtype: float64
